# NB_bishop_ch04_figures

**Bishop Chapter 4 — Key Figures: Basis Functions, Maximum Likelihood, Regularization, Bayesian Linear Regression, Predictive Distribution & Evidence Approximation**

This notebook generates pedagogical matplotlib figures for Bishop Chapter 4 on linear models for regression.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_04/NB_bishop_ch04_figures.ipynb)

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
import os

# ── colour palette ──────────────────────────────────────────────
COLORS = {
    'blue':  '#1A3A6E',
    'red':   '#CD0000',
    'green': '#2E7D32',
    'amber': '#B5853F',
}

# ── global rcParams ─────────────────────────────────────────────
matplotlib.rcParams['figure.facecolor']   = 'none'
matplotlib.rcParams['axes.facecolor']     = 'none'
matplotlib.rcParams['savefig.facecolor']  = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid']          = False
matplotlib.rcParams['legend.loc']         = 'upper center'
matplotlib.rcParams['legend.framealpha']  = 0.0

CHART_DIR = '../../charts'
os.makedirs(CHART_DIR, exist_ok=True)

def save_fig(fig, name, chart_dir=CHART_DIR):
    """Save figure with transparent background, no grid, legend outside bottom."""
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            ncol = min(len(legend.get_texts()), 4)
            legend._ncols = ncol
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

print('Setup complete.')

## Figure 4.1 — Basis Functions (Polynomial, Gaussian, Sigmoidal)

In [ ]:
x = np.linspace(-1, 1, 300)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# ── Panel 1: Polynomial basis functions ─────────────────────
ax = axes[0]
cols_poly = [COLORS['blue'], COLORS['red'], COLORS['green'],
             COLORS['amber'], '#8E24AA', '#00838F']
for j in range(6):
    ax.plot(x, x**j, lw=2, color=cols_poly[j], label=f'$x^{j}$')
ax.set_xlabel('$x$')
ax.set_ylabel('$\\phi_j(x)$')
ax.set_title('Polynomial')
ax.legend(fontsize=8)

# ── Panel 2: Gaussian basis functions ───────────────────────
ax = axes[1]
mu_vals = np.linspace(-0.8, 0.8, 6)
s = 0.25
for j, mu in enumerate(mu_vals):
    phi = np.exp(-0.5 * ((x - mu) / s)**2)
    ax.plot(x, phi, lw=2, color=cols_poly[j],
            label=f'$\\mu={mu:.1f}$')
ax.set_xlabel('$x$')
ax.set_ylabel('$\\phi_j(x)$')
ax.set_title('Gaussian RBF')
ax.legend(fontsize=8)

# ── Panel 3: Sigmoidal basis functions ──────────────────────
ax = axes[2]
for j, mu in enumerate(mu_vals):
    phi = 1.0 / (1.0 + np.exp(-(x - mu) / 0.12))
    ax.plot(x, phi, lw=2, color=cols_poly[j],
            label=f'$\\mu={mu:.1f}$')
ax.set_xlabel('$x$')
ax.set_ylabel('$\\phi_j(x)$')
ax.set_title('Sigmoidal')
ax.legend(fontsize=8)

fig.suptitle('Basis Functions for Linear Regression', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig4_1_basis_functions')
plt.show()

## Figure 4.2 — Linear Regression with Residuals

In [ ]:
np.random.seed(42)

N = 20
x_data = np.sort(np.random.uniform(0, 1, N))
y_true_fn = lambda x: np.sin(2 * np.pi * x)
noise_std = 0.3
y_data = y_true_fn(x_data) + noise_std * np.random.randn(N)

# Polynomial fit (degree 3 — reasonable)
degree = 3
coeffs = np.polyfit(x_data, y_data, degree)
x_dense = np.linspace(0, 1, 300)
y_fit = np.polyval(coeffs, x_dense)
y_fit_at_data = np.polyval(coeffs, x_data)

fig, ax = plt.subplots(figsize=(8, 5))

# True function
ax.plot(x_dense, y_true_fn(x_dense), 'k--', lw=1, alpha=0.4,
        label='True $\\sin(2\\pi x)$')

# Fitted curve
ax.plot(x_dense, y_fit, color=COLORS['blue'], lw=2.5,
        label=f'ML fit (degree {degree})')

# Residual lines
for xi, yi, yf in zip(x_data, y_data, y_fit_at_data):
    ax.plot([xi, xi], [yi, yf], color=COLORS['red'], lw=1, alpha=0.6)

# Data points (on top)
ax.scatter(x_data, y_data, s=50, zorder=5, color='white',
           edgecolor=COLORS['blue'], linewidth=1.5, label='Observations')

# Annotate a residual
idx = 8
ax.annotate('$t_n - y(x_n, \\mathbf{w})$',
            xy=(x_data[idx], (y_data[idx] + y_fit_at_data[idx]) / 2),
            xytext=(x_data[idx] + 0.12, y_data[idx] + 0.25),
            fontsize=10, color=COLORS['red'],
            arrowprops=dict(arrowstyle='->', color=COLORS['red'], lw=1.2))

ax.set_xlabel('$x$')
ax.set_ylabel('$t$')
ax.set_title('Maximum Likelihood Linear Regression')
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig4_2_linear_regression')
plt.show()

## Figure 4.3 — Effect of Regularization on Regression

In [ ]:
np.random.seed(42)

N = 15
x_data = np.linspace(0, 1, N)
y_data = np.sin(2 * np.pi * x_data) + 0.25 * np.random.randn(N)
x_dense = np.linspace(0, 1, 300)

M = 11  # high-degree polynomial

def design_matrix(x, m):
    return np.vander(x, m + 1, increasing=True)

Phi = design_matrix(x_data, M)
Phi_dense = design_matrix(x_dense, M)

cases = [
    (0,    '$\\lambda = 0$ (overfitting)',     COLORS['red']),
    (1e-6, '$\\lambda = 10^{-6}$ (good fit)',  COLORS['green']),
    (1.0,  '$\\lambda = 1$ (underfitting)',     COLORS['blue']),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (lam, title, col) in zip(axes, cases):
    I = np.eye(M + 1)
    w = np.linalg.solve(Phi.T @ Phi + lam * I, Phi.T @ y_data)
    y_fit = Phi_dense @ w

    ax.plot(x_dense, np.sin(2 * np.pi * x_dense), 'k--', lw=1, alpha=0.4,
            label='True $\\sin(2\\pi x)$')
    ax.plot(x_dense, y_fit, color=col, lw=2.5, label='Fit')
    ax.scatter(x_data, y_data, s=50, zorder=5, color='white',
               edgecolor=COLORS['blue'], linewidth=1.5, label='Data')
    ax.set_ylim(-2.0, 2.0)
    ax.set_xlabel('$x$')
    ax.set_ylabel('$t$')
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle('Effect of Regularization (Ridge / Weight Decay)', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig4_3_regularized_regression')
plt.show()

## Figure 4.6 — Sequential Bayesian Linear Regression

In [ ]:
np.random.seed(7)

# True parameters for y = w0 + w1*x
w0_true, w1_true = -0.3, 0.5
beta = 25.0   # noise precision (1/sigma^2)
alpha = 2.0   # prior precision

N_total = 25
x_all = np.random.uniform(-1, 1, N_total)
y_all = w0_true + w1_true * x_all + np.random.randn(N_total) / np.sqrt(beta)

# Stages to show: prior (0 pts), 1 pt, 5 pts, 20 pts
stages = [0, 1, 5, 20]
stage_labels = ['Prior', 'After 1 point', 'After 5 points', 'After 20 points']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

w0_range = np.linspace(-1, 1, 150)
w1_range = np.linspace(-1, 1, 150)
W0, W1 = np.meshgrid(w0_range, w1_range)
pos = np.dstack((W0, W1))

x_plot = np.linspace(-1, 1, 200)

for col, n_pts in enumerate(stages):
    # ── Bayesian update ─────────────────────────────────────
    # Prior: N(0, alpha^{-1} I)
    m0 = np.zeros(2)
    S0 = (1.0 / alpha) * np.eye(2)

    if n_pts > 0:
        x_obs = x_all[:n_pts]
        y_obs = y_all[:n_pts]
        Phi_obs = np.column_stack([np.ones(n_pts), x_obs])  # [1, x]
        S_N_inv = alpha * np.eye(2) + beta * Phi_obs.T @ Phi_obs
        S_N = np.linalg.inv(S_N_inv)
        m_N = S_N @ (beta * Phi_obs.T @ y_obs)
    else:
        S_N = S0
        m_N = m0

    # ── Top row: weight space (posterior contour) ───────────
    ax_w = axes[0, col]
    rv = multivariate_normal(mean=m_N, cov=S_N)
    Z = rv.pdf(pos)
    ax_w.contourf(W0, W1, Z, levels=15, cmap='Blues', alpha=0.7)
    ax_w.contour(W0, W1, Z, levels=6, colors=COLORS['blue'], linewidths=0.5)
    ax_w.plot(w0_true, w1_true, '+', ms=14, mew=3, color=COLORS['red'],
              zorder=5)
    ax_w.set_xlabel('$w_0$')
    ax_w.set_ylabel('$w_1$')
    ax_w.set_title(stage_labels[col], fontsize=11)
    ax_w.set_xlim(-1, 1)
    ax_w.set_ylim(-1, 1)
    ax_w.set_aspect('equal')

    # ── Bottom row: data space (sample lines) ──────────────
    ax_d = axes[1, col]

    # Draw samples from posterior
    n_samples = 6
    w_samples = np.random.multivariate_normal(m_N, S_N, size=n_samples)
    for ws in w_samples:
        ax_d.plot(x_plot, ws[0] + ws[1] * x_plot,
                  color=COLORS['red'], lw=1, alpha=0.5)

    if n_pts > 0:
        ax_d.scatter(x_obs, y_obs, s=50, zorder=5, color='white',
                     edgecolor=COLORS['blue'], linewidth=1.5)

    ax_d.set_xlabel('$x$')
    ax_d.set_ylabel('$y$')
    ax_d.set_xlim(-1, 1)
    ax_d.set_ylim(-1, 1)

axes[0, 0].set_ylabel('$w_1$\n\nWeight space', fontsize=10)
axes[1, 0].set_ylabel('$y$\n\nData space', fontsize=10)

fig.suptitle('Sequential Bayesian Linear Regression', fontsize=14, y=1.01)
fig.tight_layout()
save_fig(fig, 'fig4_6_bayesian_regression')
plt.show()

## Figure 4.7 — Predictive Distribution with Uncertainty Bands

In [ ]:
np.random.seed(42)

# Bayesian regression with Gaussian basis functions
N = 12
x_data = np.sort(np.random.uniform(0, 1, N))
y_true_fn = lambda x: np.sin(2 * np.pi * x)
beta_noise = 25.0
y_data = y_true_fn(x_data) + np.random.randn(N) / np.sqrt(beta_noise)

# Gaussian basis functions
n_basis = 9
mu_basis = np.linspace(0, 1, n_basis)
s_basis = 0.1

def phi_gaussian(x, mu_basis, s):
    """Design matrix with bias + Gaussian RBFs."""
    Phi = np.ones((len(x), len(mu_basis) + 1))
    for j, mu in enumerate(mu_basis):
        Phi[:, j + 1] = np.exp(-0.5 * ((x - mu) / s)**2)
    return Phi

Phi_train = phi_gaussian(x_data, mu_basis, s_basis)
M_dim = Phi_train.shape[1]

# Bayesian posterior
alpha_prior = 1.0
S_N_inv = alpha_prior * np.eye(M_dim) + beta_noise * Phi_train.T @ Phi_train
S_N = np.linalg.inv(S_N_inv)
m_N = S_N @ (beta_noise * Phi_train.T @ y_data)

# Predictive distribution on dense grid
x_pred = np.linspace(-0.05, 1.05, 300)
Phi_pred = phi_gaussian(x_pred, mu_basis, s_basis)

y_mean = Phi_pred @ m_N
y_var = np.array([1.0 / beta_noise + phi @ S_N @ phi
                  for phi in Phi_pred])
y_std = np.sqrt(y_var)

fig, ax = plt.subplots(figsize=(9, 5))

# Uncertainty bands (1-sigma, 2-sigma)
ax.fill_between(x_pred, y_mean - 2*y_std, y_mean + 2*y_std,
                alpha=0.10, color=COLORS['blue'], label='$\\pm 2\\sigma$')
ax.fill_between(x_pred, y_mean - y_std, y_mean + y_std,
                alpha=0.20, color=COLORS['blue'], label='$\\pm 1\\sigma$')

# True function
ax.plot(x_pred, y_true_fn(x_pred), 'k--', lw=1, alpha=0.4,
        label='True $\\sin(2\\pi x)$')

# Mean prediction
ax.plot(x_pred, y_mean, color=COLORS['red'], lw=2.5,
        label='Predictive mean $\\mathbb{E}[t|\\mathbf{x}]$')

# Data
ax.scatter(x_data, y_data, s=60, zorder=5, color='white',
           edgecolor=COLORS['blue'], linewidth=1.5, label='Observations')

# Annotate growing uncertainty
ax.annotate('Uncertainty grows\naway from data',
            xy=(1.03, y_mean[-1]), xytext=(0.82, 1.3),
            fontsize=10, color=COLORS['blue'], ha='center',
            arrowprops=dict(arrowstyle='->', color=COLORS['blue'], lw=1.2))

ax.set_xlabel('$x$')
ax.set_ylabel('$t$')
ax.set_title('Bayesian Predictive Distribution')
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-1.8, 1.8)
ax.legend()

fig.tight_layout()
save_fig(fig, 'fig4_7_predictive_distribution')
plt.show()

## Figure 4.5 — Evidence Approximation (Model Evidence vs Polynomial Degree)

In [ ]:
np.random.seed(42)

# Generate data from sin(2*pi*x)
N = 20
x_data = np.sort(np.random.uniform(0, 1, N))
y_true_fn = lambda x: np.sin(2 * np.pi * x)
beta_noise = 11.0   # true noise precision
y_data = y_true_fn(x_data) + np.random.randn(N) / np.sqrt(beta_noise)

def log_evidence_polynomial(x, y, degree, alpha, beta):
    """Compute log marginal likelihood for polynomial model.
    
    log p(y|alpha,beta,M) using Bishop eq. 3.86.
    """
    N = len(x)
    M = degree + 1  # number of parameters
    Phi = np.vander(x, M, increasing=True)

    # Posterior precision and covariance
    A = alpha * np.eye(M) + beta * Phi.T @ Phi
    S_N = np.linalg.inv(A)
    m_N = beta * S_N @ Phi.T @ y

    # E_mN = beta/2 ||y - Phi m_N||^2 + alpha/2 ||m_N||^2
    E_mN = (beta / 2.0) * np.sum((y - Phi @ m_N)**2) + \
            (alpha / 2.0) * np.sum(m_N**2)

    # Log evidence (Bishop eq. 3.86)
    sign, logdet_A = np.linalg.slogdet(A)
    log_ev = (M / 2.0) * np.log(alpha) + \
             (N / 2.0) * np.log(beta) - \
             E_mN - \
             0.5 * logdet_A - \
             (N / 2.0) * np.log(2 * np.pi)
    return log_ev

degrees = np.arange(0, 10)
alpha_fixed = 5e-3  # weak prior

log_evidences = [log_evidence_polynomial(x_data, y_data, d, alpha_fixed, beta_noise)
                 for d in degrees]

# Also compute test error for comparison
x_test = np.linspace(0, 1, 200)
y_test = y_true_fn(x_test)
test_rms = []
for d in degrees:
    M = d + 1
    Phi = np.vander(x_data, M, increasing=True)
    I = np.eye(M)
    w = np.linalg.solve(Phi.T @ Phi + alpha_fixed * I, Phi.T @ y_data)
    Phi_test = np.vander(x_test, M, increasing=True)
    y_pred = Phi_test @ w
    test_rms.append(np.sqrt(np.mean((y_pred - y_test)**2)))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Log evidence
ax = axes[0]
ax.bar(degrees, log_evidences, color=COLORS['blue'], alpha=0.7,
       edgecolor=COLORS['blue'], linewidth=1.2)
best_deg = degrees[np.argmax(log_evidences)]
ax.bar(best_deg, log_evidences[best_deg], color=COLORS['green'], alpha=0.9,
       edgecolor=COLORS['green'], linewidth=1.5)
ax.annotate(f'Best: degree {best_deg}',
            xy=(best_deg, log_evidences[best_deg]),
            xytext=(best_deg + 1.5, log_evidences[best_deg] - 3),
            fontsize=11, color=COLORS['green'],
            arrowprops=dict(arrowstyle='->', color=COLORS['green'], lw=1.5))
ax.set_xlabel('Polynomial Degree $M$')
ax.set_ylabel('Log Evidence $\\ln\\, p(\\mathbf{t} \\mid M)$')
ax.set_title('Model Evidence (Marginal Likelihood)')
ax.set_xticks(degrees)

# Panel 2: Test RMS for comparison
ax = axes[1]
ax.plot(degrees, test_rms, 'o-', color=COLORS['red'], lw=2, ms=8,
        markeredgecolor='k', markeredgewidth=0.8)
ax.axhline(1.0 / np.sqrt(beta_noise), ls=':', color='gray', lw=1,
           label='Noise level $\\sigma$')
best_rms_deg = degrees[np.argmin(test_rms)]
ax.plot(best_rms_deg, test_rms[best_rms_deg], 'o', ms=14,
        color=COLORS['green'], markeredgecolor='k', markeredgewidth=1.5,
        zorder=5, label=f'Best test RMS (degree {best_rms_deg})')
ax.set_xlabel('Polynomial Degree $M$')
ax.set_ylabel('Test RMS Error')
ax.set_title('Test Error vs Model Complexity')
ax.set_xticks(degrees)
ax.legend()

fig.suptitle('Evidence Approximation: Automatic Model Selection', fontsize=13, y=1.02)
fig.tight_layout()
save_fig(fig, 'fig4_5_evidence_approximation')
plt.show()

In [ ]:
print('All Chapter 4 figures generated.')